In [2]:
# Cell 1: Import all libraries we need
import requests
from bs4 import BeautifulSoup
import feedparser
import pandas as pd
import time
from datetime import datetime

print("✅ All imports successful!")

✅ All imports successful!


In [44]:
# ============================================================
# CELL 2: Load Bord Bia Members from CSV
# (Place bord_bia_members.csv in same folder as your notebook)
# ============================================================
# Cell 2 (UPDATED)
df_members = pd.read_csv("combined_companies.csv")
print(f"✅ Loaded {len(df_members)} Bord Bia Origin Green companies")
print(f"\nSectors covered:")
print(df_members["sector"].value_counts().to_string())
print(f"\nFirst 5 companies:")
print(df_members[["company_name","county","sector"]].head())

# Convert to list of dicts (same format as before)
companies = df_members.to_dict("records")

✅ Loaded 26 Bord Bia Origin Green companies

Sectors covered:
sector
Food            10
Dairy            5
Horticulture     3
Seafood          3
Beverage         3
Bakery           2

First 5 companies:
       company_name     county        sector
0        Flahavan's  Waterford          Food
1           Glenisk     Offaly         Dairy
2  Ballymaloe Foods       Cork          Food
3          Keelings     Dublin  Horticulture
4           Folláin       Cork          Food


In [45]:
# ============================================================
# CELL 3: Website Tester (same as before, no changes needed)
# ============================================================
def test_website(company):
    esg_keywords = [
        "sustainability", "sustainable", "environment", "carbon",
        "green", "ethical", "community", "responsible", "esg",
        "climate", "organic", "renewable", "emissions", "wellbeing"
    ]
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(company["website"], headers=headers, timeout=10, verify=False)
        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text().lower()
        found_keywords = [word for word in esg_keywords if word in page_text]
        return {
            "website_accessible" : True,
            "esg_keyword_count"  : len(found_keywords),
            "keywords_found"     : ", ".join(found_keywords) if found_keywords else "none"
        }
    except Exception as e:
        return {
            "website_accessible" : False,
            "esg_keyword_count"  : 0,
            "keywords_found"     : f"error: {str(e)[:60]}"
        }

print("✅ Website tester ready")

✅ Website tester ready


In [46]:
# ============================================================
# CELL 4: News Tester (same as before)
# ============================================================
def test_google_news(company):
    try:
        query = company["company_name"].replace(" ", "+")
        url = (
            f"https://news.google.com/rss/search"
            f"?q={query}&hl=en-IE&gl=IE&ceid=IE:en"
        )
        feed = feedparser.parse(url)
        articles = feed.entries
        if articles:
            return {
                "news_count"      : len(articles),
                "latest_headline" : articles[0].title,
                "latest_date"     : articles[0].get("published", "unknown")
            }
        return {"news_count": 0, "latest_headline": "no articles found", "latest_date": "N/A"}
    except Exception as e:
        return {"news_count": 0, "latest_headline": f"error: {str(e)[:60]}", "latest_date": "N/A"}

print("✅ News tester ready")

✅ News tester ready


In [47]:
# ============================================================
# CELL 5: Run Full Scaled Pilot on ALL companies
# This will take ~2-3 minutes due to time.sleep(1)
# ============================================================
results = []

print(f"🚀 Starting scaled pilot — {len(companies)} companies\n")

for i, company in enumerate(companies, 1):
    print(f"[{i}/{len(companies)}] Testing: {company['company_name']}...")

    web_result  = test_website(company)
    news_result = test_google_news(company)

    row = {
        # Company info
        "company_name"       : company["company_name"],
        "county"             : company["county"],
        "sector"             : company["sector"],
        "website"            : company["website"],
        # Bord Bia - already known from CSV (ground truth)
        "bord_bia_verified"  : company["bord_bia_gold_2025"],
        # Website results
        "website_accessible" : web_result["website_accessible"],
        "esg_keyword_count"  : web_result["esg_keyword_count"],
        "keywords_found"     : web_result["keywords_found"],
        # News results
        "news_count"         : news_result["news_count"],
        "latest_headline"    : news_result["latest_headline"],
    }
    results.append(row)
    time.sleep(1)

df_results = pd.DataFrame(results)

🚀 Starting scaled pilot — 26 companies

[1/26] Testing: Flahavan's...
[2/26] Testing: Glenisk...
[3/26] Testing: Ballymaloe Foods...
[4/26] Testing: Keelings...
[5/26] Testing: Folláin...
[6/26] Testing: Glenilen Farm...
[7/26] Testing: Killowen Farm...
[8/26] Testing: Sofrimar...
[9/26] Testing: Regan Organic Produce...
[10/26] Testing: Kilbeggan Organic Foods...
[11/26] Testing: Connemara Organic Seaweeds...
[12/26] Testing: Builin Blasta...
[13/26] Testing: Durrus Cheese...
[14/26] Testing: Clonakilty Food Co...
[15/26] Testing: Country Crest...
[16/26] Testing: Carbery...
[17/26] Testing: Bread 41...
[18/26] Testing: Cloud Picker Coffee...
[19/26] Testing: Cream of the Crop...
[20/26] Testing: The Naked Collective...
[21/26] Testing: Coffeeangel...
[22/26] Testing: The Happy Pear...
[23/26] Testing: Fiid...
[24/26] Testing: Wilde Irish Chocolates...
[25/26] Testing: Ballycotton Seafood...
[26/26] Testing: GIY Ireland...


In [48]:
# ============================================================
# CELL 6: Gap Score + Final Report
# ============================================================
def calculate_gap_score(esg_keyword_count, bord_bia_verified, news_count):
    score = 0
    if esg_keyword_count >= 3:
        score += 2
    elif esg_keyword_count >= 1:
        score += 1
    else:
        score -= 1
    if not bord_bia_verified:
        score += 2
    else:
        score -= 2
    if esg_keyword_count >= 3 and news_count < 50:
        score += 1
    if score >= 3:
        return score, "🔴 HIGH RISK"
    elif score >= 1:
        return score, "🟡 MEDIUM RISK"
    else:
        return score, "🟢 LOW RISK"

gap_scores, risk_levels = [], []
for _, row in df_results.iterrows():
    score, risk = calculate_gap_score(
        row["esg_keyword_count"],
        row["bord_bia_verified"],
        row["news_count"]
    )
    gap_scores.append(score)
    risk_levels.append(risk)

df_results["gap_score"]  = gap_scores
df_results["risk_level"] = risk_levels

# Save to CSV
filename = f"scaled_pilot_report_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_results.to_csv(filename, index=False)

# Print final report
print("\n" + "="*60)
print("🌿 SCALED ESG PILOT REPORT — IRISH FOOD SMEs")
print("="*60)
print(f"Companies analysed      : {len(df_results)}")
print(f"Websites accessible     : {df_results['website_accessible'].sum()}/{len(df_results)}")
print(f"With ESG keywords       : {(df_results['esg_keyword_count'] > 0).sum()}/{len(df_results)}")
print(f"Found in news           : {(df_results['news_count'] > 0).sum()}/{len(df_results)}")
print(f"Bord Bia Gold Verified  : {df_results['bord_bia_verified'].sum()}/{len(df_results)}")
print(f"\n--- RISK BREAKDOWN ---")
print(f"🔴 High Risk   : {(df_results['risk_level'] == '🔴 HIGH RISK').sum()} companies")
print(f"🟡 Medium Risk : {(df_results['risk_level'] == '🟡 MEDIUM RISK').sum()} companies")
print(f"🟢 Low Risk    : {(df_results['risk_level'] == '🟢 LOW RISK').sum()} companies")
print(f"\n--- SECTOR BREAKDOWN ---")
print(df_results.groupby("sector")["esg_keyword_count"].mean().round(1).to_string())
print(f"\n--- FULL RESULTS ---")
print(df_results[[
    "company_name", "sector", "esg_keyword_count",
    "news_count", "bord_bia_verified", "gap_score", "risk_level"
]].to_string(index=False))
print(f"\n✅ Report saved to: {filename}")


🌿 SCALED ESG PILOT REPORT — IRISH FOOD SMEs
Companies analysed      : 26
Websites accessible     : 19/26
With ESG keywords       : 14/26
Found in news           : 26/26
Bord Bia Gold Verified  : 16/26

--- RISK BREAKDOWN ---
🔴 High Risk   : 5 companies
🟡 Medium Risk : 5 companies
🟢 Low Risk    : 16 companies

--- SECTOR BREAKDOWN ---
sector
Bakery          2.0
Beverage        2.0
Dairy           1.6
Food            1.2
Horticulture    0.7
Seafood         0.3

--- FULL RESULTS ---
              company_name       sector  esg_keyword_count  news_count  bord_bia_verified  gap_score    risk_level
                Flahavan's         Food                  2         100               True         -1    🟢 LOW RISK
                   Glenisk        Dairy                  1         100               True         -1    🟢 LOW RISK
          Ballymaloe Foods         Food                  1          66               True         -1    🟢 LOW RISK
                  Keelings Horticulture               

In [49]:
# Cell A: Load FinBERT
# This downloads the FinBERT model the first time (~500MB)
# After that it's cached locally — no re-download needed
#
# ANALOGY: Think of this like hiring a specialist consultant.
# First time = they travel to your office (slow, one-time)
# Every time after = they're already there (instant)

from transformers import pipeline

print("⏳ Loading FinBERT... (may take 1-2 mins first time)")

finbert = pipeline(
    task="text-classification",
    model="ProsusAI/finbert",  # ESG-trained model on HuggingFace
    top_k=None                 # return ALL 3 scores (pos/neg/neutral)
                               # not just the top one
)

print("✅ FinBERT loaded and ready!")

# --- Quick test to make sure it works ---
test_sentence = "The company is committed to reducing carbon emissions and sustainable packaging."
result = finbert(test_sentence)
print(f"\n🧪 Test sentence: '{test_sentence}'")
print(f"📊 FinBERT result: {result}")

/Users/sundusafreen/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading FinBERT... (may take 1-2 mins first time)


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 72746.15it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FinBERT loaded and ready!

🧪 Test sentence: 'The company is committed to reducing carbon emissions and sustainable packaging.'
📊 FinBERT result: [[{'label': 'positive', 'score': 0.5310325622558594}, {'label': 'neutral', 'score': 0.46010762453079224}, {'label': 'negative', 'score': 0.00885977316647768}]]


In [50]:
# Cell B: Function to analyse sentiment of ANY text using FinBERT
#
# ANALOGY: Imagine giving FinBERT a stack of newspaper clippings
# about a company. It reads each one and gives it a mood score.
# We then average all scores to get one overall sentiment picture.

def analyse_sentiment(texts):
    """
    Takes a list of texts (headlines or articles),
    runs FinBERT on each, returns average scores.
    
    Example input: ["Company reduces emissions", "Profits drop"]
    Example output: {"positive": 0.4, "neutral": 0.5, "negative": 0.1}
    """
    if not texts:
        return {"positive": 0.0, "neutral": 1.0, "negative": 0.0}
    
    pos_scores  = []
    neu_scores  = []
    neg_scores  = []
    
    for text in texts:
        # FinBERT has a 512 token limit
        # We truncate long texts to avoid errors
        text = str(text)[:512]
        
        try:
            result = finbert(text)[0]  # list of 3 dicts
            
            # Extract each score by label
            for item in result:
                if item["label"] == "positive":
                    pos_scores.append(item["score"])
                elif item["label"] == "neutral":
                    neu_scores.append(item["score"])
                elif item["label"] == "negative":
                    neg_scores.append(item["score"])
                    
        except Exception as e:
            print(f"   ⚠️ Skipped one text: {str(e)[:50]}")
            continue
    
    # Average all scores
    # Think of it like averaging your grades across all subjects
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    avg_neu = sum(neu_scores) / len(neu_scores) if neu_scores else 0.0
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    
    return {
        "positive" : round(avg_pos, 4),
        "neutral"  : round(avg_neu, 4),
        "negative" : round(avg_neg, 4)
    }

# --- Quick test on two contrasting headlines ---
good_news = ["Company launches zero-waste packaging initiative",
             "Irish food brand wins sustainability award"]

bad_news  = ["Company fined for misleading green claims",
             "Environmental violations found at food plant"]

print("✅ Testing on GOOD news:")
print(analyse_sentiment(good_news))

print("\n✅ Testing on BAD news:")
print(analyse_sentiment(bad_news))

✅ Testing on GOOD news:
{'positive': 0.5403, 'neutral': 0.4445, 'negative': 0.0152}

✅ Testing on BAD news:
{'positive': 0.0432, 'neutral': 0.126, 'negative': 0.8308}


In [51]:
# Cell C: Fetch actual article text from news URLs
# and run FinBERT on both headlines AND article content
#
# ANALOGY: So far we've been judging books by their covers
# (headlines only). Now we're actually reading the first
# few pages of each book too (article text).

def fetch_article_text(url, max_chars=1000):
    """
    Visits a news article URL and extracts the main text.
    We limit to 1000 chars to stay within FinBERT's limit
    and keep things fast.
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers, timeout=8, verify=False)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extract paragraph text — same technique as website scraper
        paragraphs = soup.find_all("p")
        text = " ".join([p.get_text(strip=True) for p in paragraphs[:5]])
        
        return text[:max_chars] if text else ""
        
    except Exception:
        return ""  # if article fetch fails, return empty string

def get_news_sentiment(company, max_articles=5):
    """
    For a given company:
    1. Fetch up to 5 news articles via Google News RSS
    2. Collect headlines AND article text
    3. Run FinBERT on everything combined
    4. Return sentiment scores + Walk Score (0-10)
    
    ANALOGY: Like hiring 5 journalists to investigate a company,
    read their articles, and give you an overall reputation score.
    """
    try:
        # Step 1: Get news articles from Google News RSS
        query = company["company_name"].replace(" ", "+")
        url = (
            f"https://news.google.com/rss/search"
            f"?q={query}&hl=en-IE&gl=IE&ceid=IE:en"
        )
        feed = feedparser.parse(url)
        articles = feed.entries[:max_articles]  # limit to 5
        
        if not articles:
            return {
                "walk_positive"  : 0.0,
                "walk_neutral"   : 1.0,
                "walk_negative"  : 0.0,
                "walk_score"     : 5.0,  # neutral if no news
                "articles_used"  : 0
            }
        
        # Step 2: Collect headlines + article text
        all_texts = []
        
        for article in articles:
            # Always add headline
            headline = article.get("title", "")
            if headline:
                all_texts.append(headline)
            
            # Try to fetch article body text too
            article_url = article.get("link", "")
            if article_url:
                body_text = fetch_article_text(article_url)
                if body_text:
                    all_texts.append(body_text)
        
        # Step 3: Run FinBERT on everything
        print(f"   🤖 Running FinBERT on {len(all_texts)} texts...")
        scores = analyse_sentiment(all_texts)
        
        # Step 4: Convert to Walk Score (0-10)
        # Formula: starts at 5 (neutral baseline)
        # positive sentiment pushes it UP toward 10
        # negative sentiment pushes it DOWN toward 0
        #
        # Think of it like a reputation thermometer:
        # 10 = glowing reputation
        #  5 = neutral/mixed
        #  0 = terrible reputation
        walk_score = round(
            5
            + (scores["positive"] * 5)   # max +5 if all positive
            - (scores["negative"] * 5),  # max -5 if all negative
            2
        )
        # Clamp between 0 and 10
        walk_score = max(0.0, min(10.0, walk_score))
        
        return {
            "walk_positive" : scores["positive"],
            "walk_neutral"  : scores["neutral"],
            "walk_negative" : scores["negative"],
            "walk_score"    : walk_score,
            "articles_used" : len(articles)
        }
        
    except Exception as e:
        return {
            "walk_positive" : 0.0,
            "walk_neutral"  : 1.0,
            "walk_negative" : 0.0,
            "walk_score"    : 5.0,
            "articles_used" : 0
        }

# --- Quick test on ONE company first ---
print(f"🔍 Testing sentiment for: {companies[0]['company_name']}")
result = get_news_sentiment(companies[0])
print(f"  Positive score  : {result['walk_positive']}")
print(f"  Neutral score   : {result['walk_neutral']}")
print(f"  Negative score  : {result['walk_negative']}")
print(f"  Walk Score      : {result['walk_score']} / 10")
print(f"  Articles used   : {result['articles_used']}")

🔍 Testing sentiment for: Flahavan's
   🤖 Running FinBERT on 10 texts...
  Positive score  : 0.0672
  Neutral score   : 0.8639
  Negative score  : 0.069
  Walk Score      : 4.99 / 10
  Articles used   : 5


In [52]:
# Cell D: Run full sentiment analysis on all 26 companies
# and combine with existing pilot results
#
# This is the most computationally expensive cell —
# we're running FinBERT on ~260 texts total (10 per company)

sentiment_results = []

print(f"🚀 Running FinBERT sentiment on all {len(companies)} companies")
print(f"⏳ Estimated time: 8-12 minutes — please wait...\n")

for i, company in enumerate(companies, 1):
    print(f"[{i}/{len(companies)}] {company['company_name']}...")
    
    result = get_news_sentiment(company)
    
    row = {
        "company_name"   : company["company_name"],
        "group"          : company["group"],
        "bord_bia"       : company["bord_bia_gold_2025"],
        "walk_positive"  : result["walk_positive"],
        "walk_neutral"   : result["walk_neutral"],
        "walk_negative"  : result["walk_negative"],
        "walk_score"     : result["walk_score"],
        "articles_used"  : result["articles_used"]
    }
    
    sentiment_results.append(row)
    time.sleep(1)  # be polite between companies

# Convert to dataframe
df_sentiment = pd.DataFrame(sentiment_results)

# Save to CSV
df_sentiment.to_csv("sentiment_scores.csv", index=False)

# Print summary
print("\n" + "="*60)
print("📊 SENTIMENT ANALYSIS SUMMARY")
print("="*60)
print(f"Companies analysed       : {len(df_sentiment)}")
print(f"Avg Walk Score (all)     : {df_sentiment['walk_score'].mean():.2f}/10")

# Compare verified vs non-verified
verified     = df_sentiment[df_sentiment["bord_bia"] == True]
not_verified = df_sentiment[df_sentiment["bord_bia"] == False]

print(f"\nAvg Walk Score (Bord Bia Verified)  : {verified['walk_score'].mean():.2f}/10")
print(f"Avg Walk Score (Not Verified)        : {not_verified['walk_score'].mean():.2f}/10")

print("\n--- FULL RESULTS ---")
print(df_sentiment[[
    "company_name", "group", "walk_positive",
    "walk_negative", "walk_score"
]].to_string(index=False))

print(f"\n✅ Saved to sentiment_scores.csv")

🚀 Running FinBERT sentiment on all 26 companies
⏳ Estimated time: 8-12 minutes — please wait...

[1/26] Flahavan's...
   🤖 Running FinBERT on 10 texts...
[2/26] Glenisk...
   🤖 Running FinBERT on 10 texts...
[3/26] Ballymaloe Foods...
   🤖 Running FinBERT on 10 texts...
[4/26] Keelings...
   🤖 Running FinBERT on 10 texts...
[5/26] Folláin...
   🤖 Running FinBERT on 10 texts...
[6/26] Glenilen Farm...
   🤖 Running FinBERT on 10 texts...
[7/26] Killowen Farm...
   🤖 Running FinBERT on 10 texts...
[8/26] Sofrimar...
   🤖 Running FinBERT on 10 texts...
[9/26] Regan Organic Produce...
   🤖 Running FinBERT on 10 texts...
[10/26] Kilbeggan Organic Foods...
   🤖 Running FinBERT on 10 texts...
[11/26] Connemara Organic Seaweeds...
   🤖 Running FinBERT on 10 texts...
[12/26] Builin Blasta...
   🤖 Running FinBERT on 10 texts...
[13/26] Durrus Cheese...
   🤖 Running FinBERT on 10 texts...
[14/26] Clonakilty Food Co...
   🤖 Running FinBERT on 10 texts...
[15/26] Country Crest...
   🤖 Running FinBER

In [56]:
# Cell E (UPDATED): Better Talk Score using min-max normalisation
# This puts Talk and Walk on truly comparable scales
#
# ANALOGY: Imagine converting both temperatures from
# different thermometers (one Celsius, one Fahrenheit)
# to the same scale before comparing them.

# ============================================================
# STEP 1: Build Talk Score with min-max normalisation
# ============================================================
df_talk = pd.DataFrame(results)

# Add group column from df_members
df_talk = df_talk.merge(
    df_members[["company_name", "group"]],
    on="company_name",
    how="left"
)

# Min-max normalise keyword count to 0-10
# Formula: (x - min) / (max - min) * 10
def minmax_normalise(series):
    mn = series.min()
    mx = series.max()
    if mx == mn:
        return series * 0 + 5   # all same → neutral
    return ((series - mn) / (mx - mn)) * 10

df_talk["talk_score"] = minmax_normalise(
    df_talk["esg_keyword_count"]
).round(2)

# ============================================================
# STEP 2: Merge Talk + Walk
# ============================================================
df_final = df_talk[[
    "company_name", "group", "bord_bia_verified",
    "esg_keyword_count", "talk_score"
]].merge(
    df_sentiment[["company_name", "walk_score",
                  "walk_positive", "walk_negative"]],
    on="company_name"
)

# Also normalise Walk Score to same 0-10 scale
df_final["walk_score_norm"] = minmax_normalise(
    df_final["walk_score"]
).round(2)

# ============================================================
# STEP 3: Gap Score on normalised scales
# ============================================================
def final_gap_score(talk, walk):
    if talk == 0: return 0.0
    return round(max(0, (talk - walk) / talk), 3)

def risk_level(gap):
    if gap >= 0.40:    return "🔴 HIGH RISK"
    elif gap >= 0.20:  return "🟡 MEDIUM RISK"
    elif gap >= 0.05:  return "🟢 LOW RISK"
    else:              return "✅ CREDIBLE"

df_final["gap_score"] = df_final.apply(
    lambda r: final_gap_score(
        r["talk_score"], r["walk_score_norm"]
    ), axis=1
)
df_final["risk_level"] = df_final["gap_score"].apply(risk_level)
df_final = df_final.sort_values(
    "gap_score", ascending=False
).reset_index(drop=True)

# ============================================================
# STEP 4: Print Final Report
# ============================================================
print("=" * 70)
print("🌿 GREENWASHING RISK REPORT — IRISH FOOD SMEs (NORMALISED)")
print("   Talk Score = ESG keyword density (min-max normalised 0-10)")
print("   Walk Score = FinBERT news sentiment (min-max normalised 0-10)")
print("   Gap Score  = Relative gap between Talk and Walk")
print("=" * 70)

verified     = df_final[df_final["bord_bia_verified"] == True]
not_verified = df_final[df_final["bord_bia_verified"] == False]

print(f"\n📊 SUMMARY")
print(f"Total companies : {len(df_final)}")
print(f"🔴 High Risk    : {(df_final['risk_level'] == '🔴 HIGH RISK').sum()}")
print(f"🟡 Medium Risk  : {(df_final['risk_level'] == '🟡 MEDIUM RISK').sum()}")
print(f"🟢 Low Risk     : {(df_final['risk_level'] == '🟢 LOW RISK').sum()}")
print(f"✅ Credible     : {(df_final['risk_level'] == '✅ CREDIBLE').sum()}")

print(f"\n📊 GROUP COMPARISON")
print(f"Avg Gap — Bord Bia Verified : {verified['gap_score'].mean():.3f}")
print(f"Avg Gap — Not Verified      : {not_verified['gap_score'].mean():.3f}")

print(f"\n{'Company':<28} {'Talk':>5} {'Walk':>5} {'Gap':>6} "
      f"{'Risk':<14} {'Group'}")
print("-" * 78)
for _, row in df_final.iterrows():
    print(
        f"{row['company_name']:<28} "
        f"{row['talk_score']:>5.2f} "
        f"{row['walk_score_norm']:>5.2f} "
        f"{row['gap_score']:>6.3f} "
        f"{row['risk_level']:<14} "
        f"{row['group']}"
    )

df_final.to_csv("final_greenwashing_report.csv", index=False)
print(f"\n✅ Saved to: final_greenwashing_report.csv")

🌿 GREENWASHING RISK REPORT — IRISH FOOD SMEs (NORMALISED)
   Talk Score = ESG keyword density (min-max normalised 0-10)
   Walk Score = FinBERT news sentiment (min-max normalised 0-10)
   Gap Score  = Relative gap between Talk and Walk

📊 SUMMARY
Total companies : 26
🔴 High Risk    : 4
🟡 Medium Risk  : 3
🟢 Low Risk     : 0
✅ Credible     : 19

📊 GROUP COMPARISON
Avg Gap — Bord Bia Verified : 0.122
Avg Gap — Not Verified      : 0.163

Company                       Talk  Walk    Gap Risk           Group
------------------------------------------------------------------------------
The Happy Pear                8.33  0.00  1.000 🔴 HIGH RISK    Not Verified
Keelings                      3.33  0.29  0.913 🔴 HIGH RISK    Verified
Builin Blasta                 1.67  0.87  0.479 🔴 HIGH RISK    Verified
Cloud Picker Coffee           6.67  3.99  0.402 🔴 HIGH RISK    Not Verified
Carbery                      10.00  6.92  0.308 🟡 MEDIUM RISK  Verified
Flahavan's                    3.33  2.50  0.24

In [63]:
# Cell F: LLM-Based ESG Claim Extractor using Groq
#
# ANALOGY: Previously we hired a word counter to scan documents.
# Now we're hiring an experienced ESG auditor who actually reads
# and judges the quality of every sustainability claim they find.

import json
from groq import Groq

# ============================================================
# SETUP
# ============================================================
GROQ_API_KEY = "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"  # paste your key here
client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client ready")

# ============================================================
# THE PROMPT — this is the heart of Version 2
# Think of this as the briefing document you give
# to your ESG auditor before they read each company page
# ============================================================
SYSTEM_PROMPT = """
You are an expert ESG auditor specialising in greenwashing detection 
for Irish Food SMEs. Your job is to read company website text and 
evaluate the quality of their sustainability claims.

You must respond ONLY with a valid JSON object — no preamble, 
no explanation, no markdown. Just raw JSON.

Evaluate claims on these dimensions:
- Specificity: Are claims vague ("we care about environment") 
  or specific ("reduced emissions by 34% since 2019")?
- Credibility: Are claims backed by evidence, targets, 
  certifications, or third-party verification?
- Greenwashing signals: Watch for: vague language, 
  no measurable targets, unverified certifications,
  misleading comparisons, irrelevant claims.

Score everything from 1-10 where:
1  = no ESG claims at all
5  = generic vague claims with no evidence
10 = specific, measurable, verified claims

Return EXACTLY this JSON structure:
{
  "esg_claims_found": ["claim 1", "claim 2"],
  "claim_specificity": 0,
  "claim_credibility": 0,
  "greenwashing_signals": ["signal 1", "signal 2"],
  "strengths": ["strength 1"],
  "talk_score_v2": 0.0,
  "auditor_note": "one sentence summary"
}
"""

def extract_esg_claims_llm(company, website_text):
    """
    Sends website text to Groq/Llama and gets back
    a structured ESG audit in JSON format.
    
    ANALOGY: Like emailing a consultant the company brochure
    and getting back a structured due diligence report.
    """
    
    # If website wasn't accessible, tell the LLM
    if not website_text or len(website_text) < 50:
        return {
            "esg_claims_found"    : [],
            "claim_specificity"   : 0,
            "claim_credibility"   : 0,
            "greenwashing_signals": ["website not accessible"],
            "strengths"           : [],
            "talk_score_v2"       : 0.0,
            "auditor_note"        : "Website could not be accessed for analysis"
        }
    
    # Truncate to 2000 chars — enough context, within token limits
    truncated_text = website_text[:2000]
    
    user_message = f"""
Company: {company['company_name']}
Sector: {company.get('sector', 'Food')}
Country: Ireland

Website text to analyse:
---
{truncated_text}
---

Provide your ESG audit as JSON only.
"""
    
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",   # free tier model
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_message}
            ],
            temperature=0.1,   # low = more consistent, less creative
            max_tokens=500
        )
        
        # Extract the response text
        raw = response.choices[0].message.content.strip()
        
        # Clean up in case model added markdown backticks
        raw = raw.replace("```json", "").replace("```", "").strip()
        
        # Parse JSON
        result = json.loads(raw)
        return result
        
    except json.JSONDecodeError:
        # LLM returned invalid JSON — return neutral score
        return {
            "esg_claims_found"    : [],
            "claim_specificity"   : 5,
            "claim_credibility"   : 5,
            "greenwashing_signals": ["could not parse LLM response"],
            "strengths"           : [],
            "talk_score_v2"       : 5.0,
            "auditor_note"        : "Parse error — manual review needed"
        }
    except Exception as e:
        return {
            "esg_claims_found"    : [],
            "claim_specificity"   : 0,
            "claim_credibility"   : 0,
            "greenwashing_signals": [f"API error: {str(e)[:50]}"],
            "strengths"           : [],
            "talk_score_v2"       : 0.0,
            "auditor_note"        : "API call failed"
        }

print("✅ LLM extractor function ready")

✅ Groq client ready
✅ LLM extractor function ready


In [64]:
# Cell G: Fetch full website text and test LLM extractor
# on ONE company before running all 26

def fetch_website_text(company):
    """
    Fetches and cleans website text for LLM analysis.
    Returns more text than our keyword scraper —
    the LLM can handle nuance, so we give it more context.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(
            company["website"],
            headers=headers,
            timeout=10,
            verify=False
        )
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Get all meaningful paragraphs
        paragraphs = soup.find_all("p")
        text = " ".join([
            p.get_text(strip=True)
            for p in paragraphs
            if len(p.get_text(strip=True)) > 40
        ])
        
        return text if text else ""
        
    except Exception:
        return ""

# --- Test on The Happy Pear (our highest risk company) ---
test_company = next(c for c in companies if c["company_name"] == "The Happy Pear")

print(f"🌐 Fetching website text for: {test_company['company_name']}")
website_text = fetch_website_text(test_company)
print(f"   Characters fetched: {len(website_text)}")

print(f"\n🤖 Sending to Groq for ESG audit...")
result = extract_esg_claims_llm(test_company, website_text)

print(f"\n{'='*50}")
print(f"📋 ESG AUDIT RESULT — {test_company['company_name']}")
print(f"{'='*50}")
print(f"Claims found      : {result['esg_claims_found']}")
print(f"Specificity       : {result['claim_specificity']}/10")
print(f"Credibility       : {result['claim_credibility']}/10")
print(f"Greenwash signals : {result['greenwashing_signals']}")
print(f"Strengths         : {result['strengths']}")
print(f"Talk Score V2     : {result['talk_score_v2']}/10")
print(f"Auditor note      : {result['auditor_note']}")

🌐 Fetching website text for: The Happy Pear
   Characters fetched: 4783

🤖 Sending to Groq for ESG audit...

📋 ESG AUDIT RESULT — The Happy Pear
Claims found      : ['plant-based', 'nourish your body and mind', 'mindfulness practices', 'digestive health', 'gut health', 'heart health', 'plant-based nutrition', 'lifestyle tips', 'sleep quality']
Specificity       : 2/10
Credibility       : 3/10
Greenwash signals : ['no measurable targets', 'unverified certifications', 'irrelevant claims']
Strengths         : ['plant-based focus', 'emphasis on nutrition and health']
Talk Score V2     : 4.0/10
Auditor note      : The Happy Pear makes some positive ESG claims, but lacks specificity and credibility, and includes some irrelevant claims.


In [66]:
# Cell H: Run LLM ESG audit on all 26 companies
#
# ANALOGY: Sending 26 company brochures to our ESG auditor
# and getting back 26 structured due diligence reports.

llm_results = []

print(f"🚀 Running LLM ESG audit on all {len(companies)} companies\n")

for i, company in enumerate(companies, 1):
    print(f"[{i}/{len(companies)}] Auditing: {company['company_name']}...")
    
    # Step 1: Fetch website text
    website_text = fetch_website_text(company)
    
    # Step 2: Send to LLM for structured audit
    audit = extract_esg_claims_llm(company, website_text)
    
    row = {
        "company_name"        : company["company_name"],
        "group"               : company["group"],
        "bord_bia_verified"   : company["bord_bia_gold_2025"],
        "claims_found"        : " | ".join(audit["esg_claims_found"]),
        "claim_specificity"   : audit["claim_specificity"],
        "claim_credibility"   : audit["claim_credibility"],
        "greenwash_signals"   : " | ".join(audit["greenwashing_signals"]),
        "strengths"           : " | ".join(audit["strengths"]),
        "talk_score_v2"       : audit["talk_score_v2"],
        "auditor_note"        : audit["auditor_note"]
    }
    
    llm_results.append(row)
    
    # Print quick summary per company
    print(f"   Specificity: {audit['claim_specificity']}/10 | "
          f"Credibility: {audit['claim_credibility']}/10 | "
          f"Talk V2: {audit['talk_score_v2']}/10")
    
    time.sleep(0.5)  # small pause between API calls

# Convert to dataframe
df_llm = pd.DataFrame(llm_results)
df_llm.to_csv("llm_talk_scores.csv", index=False)

# Summary
print("\n" + "="*55)
print("📊 LLM ESG AUDIT SUMMARY")
print("="*55)
print(f"Companies audited        : {len(df_llm)}")
print(f"Avg Talk Score V2 (all)  : {df_llm['talk_score_v2'].mean():.2f}/10")

verified     = df_llm[df_llm["bord_bia_verified"] == True]
not_verified = df_llm[df_llm["bord_bia_verified"] == False]

print(f"Avg Talk V2 (Verified)   : {verified['talk_score_v2'].mean():.2f}/10")
print(f"Avg Talk V2 (Unverified) : {not_verified['talk_score_v2'].mean():.2f}/10")

print("\n--- FULL RESULTS ---")
print(df_llm[[
    "company_name", "group",
    "claim_specificity", "claim_credibility",
    "talk_score_v2", "auditor_note"
]].to_string(index=False))

print(f"\n✅ Saved to: llm_talk_scores.csv")

🚀 Running LLM ESG audit on all 26 companies

[1/26] Auditing: Flahavan's...
   Specificity: 8/10 | Credibility: 9/10 | Talk V2: 8.0/10
[2/26] Auditing: Glenisk...
   Specificity: 2/10 | Credibility: 6/10 | Talk V2: 4.0/10
[3/26] Auditing: Ballymaloe Foods...
   Specificity: 2/10 | Credibility: 4/10 | Talk V2: 4.0/10
[4/26] Auditing: Keelings...
   Specificity: 8/10 | Credibility: 6/10 | Talk V2: 0.67/10
[5/26] Auditing: Folláin...
   Specificity: 0/10 | Credibility: 0/10 | Talk V2: 0.0/10
[6/26] Auditing: Glenilen Farm...
   Specificity: 0/10 | Credibility: 0/10 | Talk V2: 0.0/10
[7/26] Auditing: Killowen Farm...
   Specificity: 0/10 | Credibility: 0/10 | Talk V2: 0.0/10
[8/26] Auditing: Sofrimar...
   Specificity: 2/10 | Credibility: 1/10 | Talk V2: 2.0/10
[9/26] Auditing: Regan Organic Produce...
   Specificity: 0/10 | Credibility: 0/10 | Talk V2: 0.0/10
[10/26] Auditing: Kilbeggan Organic Foods...
   Specificity: 0/10 | Credibility: 0/10 | Talk V2: 0.0/10
[11/26] Auditing: Connemara

In [67]:
# Cell I: Fix Talk Score V2 + Build Final V2 Gap Score
#
# PROBLEM: LLM sub-scores (specificity, credibility) are good
# but its final talk_score_v2 is inconsistent.
#
# FIX: We calculate Talk Score V2 ourselves using:
# talk_score_v2 = (specificity * 0.5) + (credibility * 0.5)
# Simple average of both dimensions — transparent and consistent
#
# ANALOGY: Like a teacher who grades essays on
# "argument quality" and "evidence quality" separately,
# then averages them for the final mark — rather than
# letting the student calculate their own grade.

import pandas as pd

# ============================================================
# STEP 1: Recalculate Talk Score V2 from sub-scores
# ============================================================
df_llm["talk_score_v2_fixed"] = (
    (df_llm["claim_specificity"] * 0.5) +
    (df_llm["claim_credibility"] * 0.5)
).round(2)

print("✅ Talk Score V2 recalculated from sub-scores")
print("\nComparison — LLM self-reported vs our calculation:")
print(df_llm[[
    "company_name",
    "claim_specificity",
    "claim_credibility",
    "talk_score_v2",         # LLM's own calculation
    "talk_score_v2_fixed"    # our recalculation
]].to_string(index=False))

# ============================================================
# STEP 2: Merge with Walk Score from FinBERT
# ============================================================
df_v2 = df_llm[[
    "company_name", "group", "bord_bia_verified",
    "claim_specificity", "claim_credibility",
    "talk_score_v2_fixed", "greenwash_signals", "auditor_note"
]].merge(
    df_sentiment[[
        "company_name", "walk_score",
        "walk_positive", "walk_negative"
    ]],
    on="company_name"
)

# ============================================================
# STEP 3: Normalise both scores to 0-10 scale
# ============================================================
def minmax_normalise(series):
    mn = series.min()
    mx = series.max()
    if mx == mn:
        return series * 0 + 5
    return ((series - mn) / (mx - mn)) * 10

df_v2["talk_norm"] = minmax_normalise(
    df_v2["talk_score_v2_fixed"]
).round(2)

df_v2["walk_norm"] = minmax_normalise(
    df_v2["walk_score"]
).round(2)

# ============================================================
# STEP 4: Calculate V2 Gap Score
# ============================================================
def gap_score_v2(talk, walk):
    if talk == 0: return 0.0
    return round(max(0, (talk - walk) / talk), 3)

def risk_level_v2(gap):
    if gap >= 0.40:   return "🔴 HIGH RISK"
    elif gap >= 0.20: return "🟡 MEDIUM RISK"
    elif gap >= 0.05: return "🟢 LOW RISK"
    else:             return "✅ CREDIBLE"

df_v2["gap_score_v2"] = df_v2.apply(
    lambda r: gap_score_v2(r["talk_norm"], r["walk_norm"]), axis=1
)
df_v2["risk_level_v2"] = df_v2["gap_score_v2"].apply(risk_level_v2)
df_v2 = df_v2.sort_values(
    "gap_score_v2", ascending=False
).reset_index(drop=True)

# ============================================================
# STEP 5: Print Final V2 Report
# ============================================================
print("\n" + "="*70)
print("🌿 GREENWASHING REPORT V2 — LLM + FinBERT PIPELINE")
print("   Talk Score = LLM ESG claim quality (specificity + credibility)")
print("   Walk Score = FinBERT news sentiment")
print("   Gap Score  = Relative divergence (higher = more risk)")
print("="*70)

verified     = df_v2[df_v2["bord_bia_verified"] == True]
not_verified = df_v2[df_v2["bord_bia_verified"] == False]

print(f"\n📊 SUMMARY")
print(f"Companies analysed : {len(df_v2)}")
print(f"🔴 High Risk       : {(df_v2['risk_level_v2'] == '🔴 HIGH RISK').sum()}")
print(f"🟡 Medium Risk     : {(df_v2['risk_level_v2'] == '🟡 MEDIUM RISK').sum()}")
print(f"🟢 Low Risk        : {(df_v2['risk_level_v2'] == '🟢 LOW RISK').sum()}")
print(f"✅ Credible        : {(df_v2['risk_level_v2'] == '✅ CREDIBLE').sum()}")

print(f"\n📊 GROUP COMPARISON")
print(f"Avg Gap V2 — Bord Bia Verified : {verified['gap_score_v2'].mean():.3f}")
print(f"Avg Gap V2 — Not Verified      : {not_verified['gap_score_v2'].mean():.3f}")

print(f"\n{'Company':<28} {'Talk':>5} {'Walk':>5} "
      f"{'Gap':>6} {'Risk':<14} {'Group'}")
print("-" * 78)
for _, row in df_v2.iterrows():
    print(
        f"{row['company_name']:<28} "
        f"{row['talk_norm']:>5.2f} "
        f"{row['walk_norm']:>5.2f} "
        f"{row['gap_score_v2']:>6.3f} "
        f"{row['risk_level_v2']:<14} "
        f"{row['group']}"
    )

# Save
df_v2.to_csv("final_report_v2.csv", index=False)
print(f"\n✅ Saved to: final_report_v2.csv")

✅ Talk Score V2 recalculated from sub-scores

Comparison — LLM self-reported vs our calculation:
              company_name  claim_specificity  claim_credibility  talk_score_v2  talk_score_v2_fixed
                Flahavan's                  8                  9           8.00                  8.5
                   Glenisk                  2                  6           4.00                  4.0
          Ballymaloe Foods                  2                  4           4.00                  3.0
                  Keelings                  8                  6           0.67                  7.0
                   Folláin                  0                  0           0.00                  0.0
             Glenilen Farm                  0                  0           0.00                  0.0
             Killowen Farm                  0                  0           0.00                  0.0
                  Sofrimar                  2                  1           2.00                

In [68]:
# Cell J: Side-by-side comparison of V1 vs V2
# This is your key dissertation contribution table —
# showing how LLM-based extraction changes risk classification

# ============================================================
# Merge V1 and V2 results
# ============================================================
df_v1 = df_final[["company_name", "group", "bord_bia_verified",
                   "talk_score", "walk_score_norm",
                   "gap_score", "risk_level"]].copy()

df_v1.columns = ["company_name", "group", "bord_bia_verified",
                  "talk_v1", "walk_v1",
                  "gap_v1", "risk_v1"]

df_v2_compare = df_v2[["company_name", "talk_norm", "walk_norm",
                         "gap_score_v2", "risk_level_v2"]].copy()

df_v2_compare.columns = ["company_name", "talk_v2", "walk_v2",
                           "gap_v2", "risk_v2"]

df_compare = df_v1.merge(df_v2_compare, on="company_name")
df_compare = df_compare.sort_values("gap_v2", ascending=False).reset_index(drop=True)

# ============================================================
# Flag companies that CHANGED risk classification
# ============================================================
df_compare["risk_changed"] = df_compare["risk_v1"] != df_compare["risk_v2"]
df_compare["talk_change"]  = (df_compare["talk_v2"] - df_compare["talk_v1"]).round(2)

# ============================================================
# Print comparison report
# ============================================================
print("=" * 85)
print("📊 V1 vs V2 COMPARISON — IMPACT OF LLM CLAIM EXTRACTION")
print("   V1: Keyword counting | V2: LLM semantic audit (Llama 3.1)")
print("=" * 85)

changed = df_compare[df_compare["risk_changed"] == True]
print(f"\n🔄 Companies that CHANGED risk classification: {len(changed)}/26")
print(f"   These are the cases where LLM made the biggest difference:\n")

for _, row in changed.iterrows():
    direction = "⬆️ escalated" if row["gap_v2"] > row["gap_v1"] else "⬇️ de-escalated"
    print(f"   {row['company_name']:<28} "
          f"{row['risk_v1']:<16} → {row['risk_v2']:<16} {direction}")

print(f"\n{'Company':<28} {'Talk V1':>7} {'Talk V2':>7} "
      f"{'Gap V1':>7} {'Gap V2':>7} {'Risk V1':<16} {'Risk V2':<16} {'Group'}")
print("-" * 105)

for _, row in df_compare.iterrows():
    changed_marker = " ←" if row["risk_changed"] else ""
    print(
        f"{row['company_name']:<28} "
        f"{row['talk_v1']:>7.2f} "
        f"{row['talk_v2']:>7.2f} "
        f"{row['gap_v1']:>7.3f} "
        f"{row['gap_v2']:>7.3f} "
        f"{row['risk_v1']:<16} "
        f"{row['risk_v2']:<16} "
        f"{row['group']}"
        f"{changed_marker}"
    )

# ============================================================
# Summary statistics
# ============================================================
print(f"\n📊 OVERALL IMPACT")
print(f"Avg Talk Score V1 : {df_compare['talk_v1'].mean():.2f}/10")
print(f"Avg Talk Score V2 : {df_compare['talk_v2'].mean():.2f}/10")
print(f"Avg Gap Score V1  : {df_compare['gap_v1'].mean():.3f}")
print(f"Avg Gap Score V2  : {df_compare['gap_v2'].mean():.3f}")

print(f"\n📊 GROUP SEPARATION")
ver   = df_compare[df_compare["bord_bia_verified"] == True]
unver = df_compare[df_compare["bord_bia_verified"] == False]
print(f"V1 gap difference (unverified - verified) : "
      f"{unver['gap_v1'].mean() - ver['gap_v1'].mean():.3f}")
print(f"V2 gap difference (unverified - verified) : "
      f"{unver['gap_v2'].mean() - ver['gap_v2'].mean():.3f}")

print(f"\n📊 RISK RECLASSIFICATION SUMMARY")
for risk in ["🔴 HIGH RISK", "🟡 MEDIUM RISK", "🟢 LOW RISK", "✅ CREDIBLE"]:
    v1_count = (df_compare["risk_v1"] == risk).sum()
    v2_count = (df_compare["risk_v2"] == risk).sum()
    change = v2_count - v1_count
    arrow = f"(+{change})" if change > 0 else f"({change})" if change < 0 else "(no change)"
    print(f"  {risk:<16} V1: {v1_count} → V2: {v2_count} {arrow}")

# Save
df_compare.to_csv("v1_vs_v2_comparison.csv", index=False)
print(f"\n✅ Saved to: v1_vs_v2_comparison.csv")

📊 V1 vs V2 COMPARISON — IMPACT OF LLM CLAIM EXTRACTION
   V1: Keyword counting | V2: LLM semantic audit (Llama 3.1)

🔄 Companies that CHANGED risk classification: 5/26
   These are the cases where LLM made the biggest difference:

   Flahavan's                   🟡 MEDIUM RISK    → 🔴 HIGH RISK      ⬆️ escalated
   Bread 41                     ✅ CREDIBLE       → 🟡 MEDIUM RISK    ⬆️ escalated
   Builin Blasta                🔴 HIGH RISK      → 🟡 MEDIUM RISK    ⬇️ de-escalated
   Carbery                      🟡 MEDIUM RISK    → 🟢 LOW RISK       ⬇️ de-escalated
   Cloud Picker Coffee          🔴 HIGH RISK      → ✅ CREDIBLE       ⬇️ de-escalated

Company                      Talk V1 Talk V2  Gap V1  Gap V2 Risk V1          Risk V2          Group
---------------------------------------------------------------------------------------------------------
The Happy Pear                  8.33    2.94   1.000   1.000 🔴 HIGH RISK      🔴 HIGH RISK      Not Verified
Keelings                        3.33   

In [69]:
# Cell K: Hybrid Pipeline — LLM + FinBERT combined
#
# ARCHITECTURE:
# ┌─────────────────────────────────────────────────┐
# │  TALK SCORE (Hybrid)                            │
# │  ├── LLM Score    (specificity + credibility)   │
# │  └── FinBERT Score (claim sentiment)            │
# │       weighted average → Hybrid Talk Score      │
# ├─────────────────────────────────────────────────│
# │  WALK SCORE                                     │
# │  └── FinBERT (news sentiment) — unchanged       │
# ├─────────────────────────────────────────────────│
# │  GAP SCORE = Talk - Walk divergence             │
# └─────────────────────────────────────────────────┘
#
# ANALOGY: Like a job candidate evaluation where:
# LLM    = HR reads the CV and scores quality
# FinBERT = Tone analyser scores how confidently 
#           claims are written
# Combined = final candidate talk score
# News sentiment = what references actually say (walk)

# ============================================================
# WEIGHTS — you can tune these
# 0.6 = trust LLM score more
# 0.4 = trust FinBERT claim sentiment somewhat
# These are tunable hyperparameters for your dissertation
# ============================================================
WEIGHT_LLM     = 0.6   # LLM specificity + credibility
WEIGHT_FINBERT = 0.4   # FinBERT claim sentiment

print(f"⚙️  Weights: LLM={WEIGHT_LLM}, FinBERT={WEIGHT_FINBERT}")
print(f"✅ These can be tuned — try 0.5/0.5 or 0.7/0.3 later\n")

# ============================================================
# STEP 1: Run FinBERT on LLM-extracted claims
# (Approach 1 — focused sentiment on claims only)
# ============================================================
def finbert_on_claims(claims_text):
    """
    Run FinBERT specifically on the claims the LLM extracted.
    This is more focused than running on the full website text.
    
    ANALOGY: Instead of asking FinBERT to read the whole essay,
    we highlight the thesis statements and ask it to judge those.
    """
    if not claims_text or claims_text.strip() == "":
        return {"positive": 0.0, "neutral": 1.0, "negative": 0.0,
                "claim_sentiment_score": 5.0}
    
    # Split pipe-separated claims into individual sentences
    claims = [c.strip() for c in claims_text.split("|") if len(c.strip()) > 5]
    
    if not claims:
        return {"positive": 0.0, "neutral": 1.0, "negative": 0.0,
                "claim_sentiment_score": 5.0}
    
    # Run FinBERT on each individual claim
    scores = analyse_sentiment(claims)  # reuse function from Cell B
    
    # Convert to 0-10 claim sentiment score
    # Same formula as Walk Score — positive pushes up, negative pushes down
    claim_sentiment_score = round(
        5 + (scores["positive"] * 5) - (scores["negative"] * 5), 2
    )
    claim_sentiment_score = max(0.0, min(10.0, claim_sentiment_score))
    
    return {
        "claim_pos"             : scores["positive"],
        "claim_neu"             : scores["neutral"],
        "claim_neg"             : scores["negative"],
        "claim_sentiment_score" : claim_sentiment_score
    }

# ============================================================
# STEP 2: Build Hybrid Talk Score
# Combines LLM quality score + FinBERT claim sentiment
# ============================================================
def hybrid_talk_score(llm_score, claim_sentiment_score,
                      weight_llm=WEIGHT_LLM,
                      weight_finbert=WEIGHT_FINBERT):
    """
    Weighted average of:
    - LLM score     (quality: specificity + credibility)
    - FinBERT score (sentiment: how confidently claims are written)
    
    ANALOGY: Like a weighted average grade —
    LLM = coursework (60%), FinBERT = exam (40%)
    """
    return round(
        (llm_score * weight_llm) +
        (claim_sentiment_score * weight_finbert),
        2
    )

# ============================================================
# STEP 3: Run full hybrid pipeline on all 26 companies
# ============================================================
hybrid_results = []

print(f"🚀 Running Hybrid Pipeline on {len(companies)} companies\n")

for i, company in enumerate(companies, 1):
    print(f"[{i}/{len(companies)}] {company['company_name']}...")
    
    # Get LLM scores from df_llm (already computed in Cell H)
    llm_row = df_llm[df_llm["company_name"] == company["company_name"]]
    
    if llm_row.empty:
        print(f"   ⚠️  No LLM data found — skipping")
        continue
    
    llm_row = llm_row.iloc[0]
    
    # --- Approach 1: FinBERT on extracted claims ---
    print(f"   🤖 Running FinBERT on extracted claims...")
    claim_scores = finbert_on_claims(str(llm_row["claims_found"]))
    
    # --- LLM quality score (already computed) ---
    llm_quality = float(llm_row["talk_score_v2_fixed"])  # from Cell I
    
    # --- Hybrid Talk Score (Approach 3) ---
    hybrid_talk = hybrid_talk_score(
        llm_quality,
        claim_scores["claim_sentiment_score"]
    )
    
    # --- Walk Score from FinBERT news (already computed) ---
    walk_row = df_sentiment[df_sentiment["company_name"] == company["company_name"]]
    walk_score = float(walk_row["walk_score"].iloc[0]) if not walk_row.empty else 5.0
    
    row = {
        "company_name"          : company["company_name"],
        "group"                 : company["group"],
        "bord_bia_verified"     : company["bord_bia_gold_2025"],
        # LLM component
        "llm_quality_score"     : llm_quality,
        "claim_specificity"     : int(llm_row["claim_specificity"]),
        "claim_credibility"     : int(llm_row["claim_credibility"]),
        # FinBERT on claims component
        "claim_sentiment_score" : claim_scores["claim_sentiment_score"],
        "claim_pos"             : claim_scores.get("claim_pos", 0),
        "claim_neg"             : claim_scores.get("claim_neg", 0),
        # Hybrid Talk Score
        "hybrid_talk_score"     : hybrid_talk,
        # Walk Score (FinBERT news)
        "walk_score"            : walk_score,
    }
    
    hybrid_results.append(row)
    time.sleep(0.3)

df_hybrid = pd.DataFrame(hybrid_results)

print(f"\n✅ Hybrid pipeline complete — {len(df_hybrid)} companies processed")

# ============================================================
# STEP 4: Normalise + Calculate Hybrid Gap Score
# ============================================================
def minmax_normalise(series):
    mn, mx = series.min(), series.max()
    if mx == mn: return series * 0 + 5
    return ((series - mn) / (mx - mn)) * 10

df_hybrid["talk_norm_h"] = minmax_normalise(
    df_hybrid["hybrid_talk_score"]
).round(2)

df_hybrid["walk_norm_h"] = minmax_normalise(
    df_hybrid["walk_score"]
).round(2)

def gap_hybrid(talk, walk):
    if talk == 0: return 0.0
    return round(max(0, (talk - walk) / talk), 3)

def risk_hybrid(gap):
    if gap >= 0.40:   return "🔴 HIGH RISK"
    elif gap >= 0.20: return "🟡 MEDIUM RISK"
    elif gap >= 0.05: return "🟢 LOW RISK"
    else:             return "✅ CREDIBLE"

df_hybrid["gap_hybrid"]  = df_hybrid.apply(
    lambda r: gap_hybrid(r["talk_norm_h"], r["walk_norm_h"]), axis=1
)
df_hybrid["risk_hybrid"] = df_hybrid["gap_hybrid"].apply(risk_hybrid)
df_hybrid = df_hybrid.sort_values(
    "gap_hybrid", ascending=False
).reset_index(drop=True)

# ============================================================
# STEP 5: Print Final Hybrid Report
# ============================================================
print("\n" + "="*75)
print("🌿 HYBRID PIPELINE REPORT — LLM + FinBERT COMBINED")
print(f"   Talk = (LLM quality × {WEIGHT_LLM}) + (FinBERT claim sentiment × {WEIGHT_FINBERT})")
print(f"   Walk = FinBERT news sentiment (unchanged)")
print("="*75)

verified     = df_hybrid[df_hybrid["bord_bia_verified"] == True]
not_verified = df_hybrid[df_hybrid["bord_bia_verified"] == False]

print(f"\n📊 SUMMARY")
print(f"🔴 High Risk   : {(df_hybrid['risk_hybrid'] == '🔴 HIGH RISK').sum()}")
print(f"🟡 Medium Risk : {(df_hybrid['risk_hybrid'] == '🟡 MEDIUM RISK').sum()}")
print(f"🟢 Low Risk    : {(df_hybrid['risk_hybrid'] == '🟢 LOW RISK').sum()}")
print(f"✅ Credible    : {(df_hybrid['risk_hybrid'] == '✅ CREDIBLE').sum()}")

print(f"\n📊 GROUP SEPARATION")
print(f"Avg Gap — Bord Bia Verified : {verified['gap_hybrid'].mean():.3f}")
print(f"Avg Gap — Not Verified      : {not_verified['gap_hybrid'].mean():.3f}")
diff_hybrid = not_verified['gap_hybrid'].mean() - verified['gap_hybrid'].mean()
print(f"Difference                  : {diff_hybrid:.3f}")

print(f"\n{'Company':<28} {'LLM':>5} {'ClmSent':>8} {'HybTalk':>8} "
      f"{'Walk':>5} {'Gap':>6} {'Risk':<14} {'Group'}")
print("-" * 90)
for _, row in df_hybrid.iterrows():
    print(
        f"{row['company_name']:<28} "
        f"{row['llm_quality_score']:>5.1f} "
        f"{row['claim_sentiment_score']:>8.2f} "
        f"{row['talk_norm_h']:>8.2f} "
        f"{row['walk_norm_h']:>5.2f} "
        f"{row['gap_hybrid']:>6.3f} "
        f"{row['risk_hybrid']:<14} "
        f"{row['group']}"
    )

# Save
df_hybrid.to_csv("hybrid_pipeline_report.csv", index=False)
print(f"\n✅ Saved to: hybrid_pipeline_report.csv")

⚙️  Weights: LLM=0.6, FinBERT=0.4
✅ These can be tuned — try 0.5/0.5 or 0.7/0.3 later

🚀 Running Hybrid Pipeline on 26 companies

[1/26] Flahavan's...
   🤖 Running FinBERT on extracted claims...
[2/26] Glenisk...
   🤖 Running FinBERT on extracted claims...
[3/26] Ballymaloe Foods...
   🤖 Running FinBERT on extracted claims...
[4/26] Keelings...
   🤖 Running FinBERT on extracted claims...
[5/26] Folláin...
   🤖 Running FinBERT on extracted claims...
[6/26] Glenilen Farm...
   🤖 Running FinBERT on extracted claims...
[7/26] Killowen Farm...
   🤖 Running FinBERT on extracted claims...
[8/26] Sofrimar...
   🤖 Running FinBERT on extracted claims...
[9/26] Regan Organic Produce...
   🤖 Running FinBERT on extracted claims...
[10/26] Kilbeggan Organic Foods...
   🤖 Running FinBERT on extracted claims...
[11/26] Connemara Organic Seaweeds...
   🤖 Running FinBERT on extracted claims...
[12/26] Builin Blasta...
   🤖 Running FinBERT on extracted claims...
[13/26] Durrus Cheese...
   🤖 Running FinB